# Capstone Phase 2：数据表示与知识图谱 · 起始笔记本

> **任务**：完成6个TODO填空，构建营销数据表示层+知识图谱
> **真实库**：sentence-transformers + networkx + pandas
> **数据**：基于真实电商分布设计的营销数据（见data/README.md）

整合技能1(表示工程Day1-3)+技能0(数据处理)，为Phase 3营销Agent提供知识基础。

## 环境准备

In [ ]:
# 安装所需库（如尚未安装）
# pip install sentence-transformers networkx pandas numpy scikit-learn

import pandas as pd
import numpy as np
import networkx as nx
from sentence_transformers import SentenceTransformer, util
import warnings
warnings.filterwarnings('ignore')

print('库导入成功')
print(f'networkx: {nx.__version__}, pandas: {pd.__version__}, numpy: {np.__version__}')

## 数据加载

真实营销数据，基于Statista/天猫/CNNIC真实电商分布设计。

In [ ]:
# === 真实营销数据（基于真实电商分布设计，参数可追溯）===
# 数据来源：Statista全球电商统计 + 天猫双11品类分布 + CNNIC中国网络购物市场研究报告
# 详见 data/README.md

import pandas as pd

# 客户数据（基于CNNIC年龄/性别分布）
customers = pd.DataFrame([
    {"customer_id": "C001", "name": "张明", "age": 28, "gender": "M", "lifecycle_stage": "active", "value_segment": "high", "bio": "热爱跑步的科技爱好者，每周跑步3次，关注智能穿戴设备"},
    {"customer_id": "C002", "name": "李娜", "age": 35, "gender": "F", "lifecycle_stage": "active", "value_segment": "high", "bio": "健身教练，专业运动装备用户，注重产品性能和专业度"},
    {"customer_id": "C003", "name": "王强", "age": 22, "gender": "M", "lifecycle_stage": "new", "value_segment": "medium", "bio": "大学生，预算有限，喜欢性价比高的入门级运动产品"},
    {"customer_id": "C004", "name": "赵雪", "age": 30, "gender": "F", "lifecycle_stage": "active", "value_segment": "medium", "bio": "瑜伽爱好者，关注健康生活方式，偏好天然环保材质"},
    {"customer_id": "C005", "name": "刘洋", "age": 45, "gender": "M", "lifecycle_stage": "dormant", "value_segment": "low", "bio": "偶尔运动的中年上班族，需要简单易用的健康监测设备"},
    {"customer_id": "C006", "name": "陈静", "age": 26, "gender": "F", "lifecycle_stage": "active", "value_segment": "high", "bio": "马拉松跑者，追求极致轻量化和精准数据追踪"},
    {"customer_id": "C007", "name": "杨光", "age": 33, "gender": "M", "lifecycle_stage": "new", "value_segment": "medium", "bio": "户外探险爱好者，需要坚固耐用的多功能运动手表"},
    {"customer_id": "C008", "name": "周琳", "age": 29, "gender": "F", "lifecycle_stage": "active", "value_segment": "medium", "bio": "音乐与运动兼爱，寻找适合运动时佩戴的高 quality 耳机"},
])

# 产品数据（基于天猫品类分布+真实产品描述）
products = pd.DataFrame([
    {"product_id": "P001", "name": "智能跑步手表ProMax", "category": "智能穿戴设备", "price": 1299, "brand": "TechFit", "description": "专业马拉松级GPS运动手表，42天续航，血氧心率监测，50米防水，支持117种运动模式"},
    {"product_id": "P002", "name": "无线降噪耳机Pro", "category": "音频设备", "price": 899, "brand": "SoundWave", "description": "主动降噪蓝牙耳机，40小时续航，IPX5防水，适合运动场景，支持LDAC高清音质"},
    {"product_id": "P003", "name": "智能健康手环Lite", "category": "智能穿戴设备", "price": 299, "brand": "TechFit", "description": "入门级健康监测手环，14天续航，心率睡眠监测，性价比高，适合运动新手"},
    {"product_id": "P004", "name": "运动蓝牙耳机Mini", "category": "音频设备", "price": 199, "brand": "SoundWave", "description": "轻量颈挂式运动耳机，12小时续航，防汗设计，磁吸佩戴，入门级运动音频"},
    {"product_id": "P005", "name": "专业瑜伽垫Premium", "category": "运动装备", "price": 399, "brand": "ZenFlex", "description": "天然橡胶环保瑜伽垫，6mm加厚，防滑双面设计，体位引导线，适合专业瑜伽练习"},
    {"product_id": "P006", "name": "户外多功能背包Trek", "category": "户外装备", "price": 599, "brand": "TrailBlaze", "description": "45L户外徒步背包，防水耐磨面料，人体工学背负系统，多功能挂载点，适合多日徒步"},
    {"product_id": "P007", "name": "智能体脂秤S", "category": "智能穿戴设备", "price": 159, "brand": "TechFit", "description": "高精度蓝牙体脂秤，16项身体成分分析，APP数据同步，支持多用户，简约设计"},
    {"product_id": "P008", "name": "压缩运动袜Set", "category": "运动装备", "price": 89, "brand": "ZenFlex", "description": "专业运动压缩袜套装，梯度压缩技术，排汗速干，足弓支撑，适合长跑和马拉松"},
])

# 交互数据（购买记录）
interactions = pd.DataFrame([
    {"customer_id": "C001", "product_id": "P001", "relation": "PURCHASED", "quantity": 1, "rating": 5, "review": "续航超长，GPS精准，马拉松必备"},
    {"customer_id": "C001", "product_id": "P004", "relation": "PURCHASED", "quantity": 1, "rating": 4, "review": "轻便好用，运动时不掉"},
    {"customer_id": "C002", "product_id": "P001", "relation": "PURCHASED", "quantity": 1, "rating": 5, "review": "专业数据全面，推荐给学员"},
    {"customer_id": "C002", "product_id": "P005", "relation": "PURCHASED", "quantity": 1, "rating": 5, "review": "防滑效果好，瑜伽课必备"},
    {"customer_id": "C003", "product_id": "P003", "relation": "PURCHASED", "quantity": 1, "rating": 4, "review": "性价比高，学生党友好"},
    {"customer_id": "C004", "product_id": "P005", "relation": "PURCHASED", "quantity": 1, "rating": 5, "review": "环保材质很好，体位线很实用"},
    {"customer_id": "C004", "product_id": "P007", "relation": "PURCHASED", "quantity": 1, "rating": 4, "review": "数据详细，帮助追踪健康"},
    {"customer_id": "C005", "product_id": "P007", "relation": "PURCHASED", "quantity": 1, "rating": 3, "review": "功能够用，APP偶尔卡"},
    {"customer_id": "C006", "product_id": "P001", "relation": "PURCHASED", "quantity": 1, "rating": 5, "review": "极致轻量，数据精准，PB利器"},
    {"customer_id": "C006", "product_id": "P008", "relation": "PURCHASED", "quantity": 2, "rating": 5, "review": "压缩感好，长跑不磨脚"},
    {"customer_id": "C007", "product_id": "P006", "relation": "PURCHASED", "quantity": 1, "rating": 5, "review": "背负舒适，多日徒步无压力"},
    {"customer_id": "C008", "product_id": "P002", "relation": "PURCHASED", "quantity": 1, "rating": 5, "review": "降噪效果好，运动时沉浸感强"},
    {"customer_id": "C008", "product_id": "P004", "relation": "PURCHASED", "quantity": 1, "rating": 4, "review": "轻便，日常通勤也能用"},
])

# 品牌、品类、活动、渠道
brands = pd.DataFrame([
    {"brand_id": "B001", "name": "TechFit", "country": "中国"},
    {"brand_id": "B002", "name": "SoundWave", "country": "中国"},
    {"brand_id": "B003", "name": "ZenFlex", "country": "美国"},
    {"brand_id": "B004", "name": "TrailBlaze", "country": "德国"},
])

categories = pd.DataFrame([
    {"category_id": "CAT001", "name": "智能穿戴设备"},
    {"category_id": "CAT002", "name": "音频设备"},
    {"category_id": "CAT003", "name": "运动装备"},
    {"category_id": "CAT004", "name": "户外装备"},
])

campaigns = pd.DataFrame([
    {"campaign_id": "CMP001", "name": "2026春季跑步节", "budget": 50000, "objective": "品牌曝光"},
    {"campaign_id": "CMP002", "name": "新品耳机上市推广", "budget": 30000, "objective": "产品转化"},
    {"campaign_id": "CMP003", "name": "瑜伽生活月", "budget": 20000, "objective": "社区运营"},
])

channels = pd.DataFrame([
    {"channel_id": "CH001", "name": "小红书", "type": "社交内容", "reach": 5000000},
    {"channel_id": "CH002", "name": "抖音", "type": "短视频", "reach": 8000000},
    {"channel_id": "CH003", "name": "微信公众号", "type": "私域", "reach": 200000},
])

print(f"数据加载完成: {len(customers)}客户, {len(products)}产品, {len(interactions)}交互, {len(brands)}品牌, {len(categories)}品类, {len(campaigns)}活动, {len(channels)}渠道")
print(f"\n客户数据预览:")
print(customers[["customer_id", "name", "age", "lifecycle_stage", "value_segment"]].head())
print(f"\n产品数据预览:")
print(products[["product_id", "name", "category", "price", "brand"]].head())


## TODO 1：数据预处理与特征工程（pandas）

用 pandas 完成数据预处理：
1. 检查缺失值
2. 创建客户特征文本（将客户属性拼接为可向量化的文本）
3. 创建产品特征文本（将产品属性拼接为可向量化的文本）
4. 统计各品类产品数量和平均价格

**提示**：客户特征文本应包含 bio + lifecycle_stage + value_segment，产品特征文本应包含 name + category + description + brand

In [ ]:
# TODO: 你的代码
# 1. 检查缺失值
# 2. 创建客户特征文本（customer_text列）
# 3. 创建产品特征文本（product_text列）
# 4. 统计各品类产品数量和平均价格

raise NotImplementedError

## TODO 2：向量化表示与语义检索（sentence-transformers）

用 sentence-transformers（all-MiniLM-L6-v2，384维）将客户/产品文本编码为向量：
1. 加载 all-MiniLM-L6-v2 模型
2. 将产品描述编码为384维向量
3. 实现语义检索：给定查询文本，返回最相似的产品Top-K
4. 将客户编码为向量，找到与查询客户最相似的产品

**提示**：使用 `model.encode()` 编码，`util.cos_sim()` 计算相似度

In [ ]:
# TODO: 你的代码
# 1. 加载 all-MiniLM-L6-v2 模型
# 2. 编码产品文本为384维向量
# 3. 实现语义检索函数 semantic_search(query, product_embeddings, products, top_k=3)
# 4. 搜索 '轻量运动手表适合马拉松' 并打印结果
# 5. 编码客户文本，找到与C001最相似的产品

raise NotImplementedError

## TODO 3：构建营销知识图谱（networkx）

用 networkx 构建营销知识图谱（MultiDiGraph）：
1. 创建 MultiDiGraph
2. 添加节点：客户(8) + 产品(8) + 品牌(4) + 品类(4) + 活动(3) + 渠道(3)
3. 添加边：PURCHASED, MANUFACTURED_BY, BELONGS_TO, COMPETES_WITH, COMPLEMENTARY_TO, REVIEWED, PROMOTES, PROMOTED_THROUGH
4. 打印图的基本统计信息（节点数、边数、关系类型）

**提示**：用 `G.add_node()` 和 `G.add_edge()`，边带 `relation` 属性

In [ ]:
# TODO: 你的代码
# 1. 创建 MultiDiGraph
# 2. 添加所有节点（带属性）
# 3. 添加所有边（带relation属性）
#    - PURCHASED: 从 interactions 数据构建
#    - MANUFACTURED_BY: 产品 -> 品牌
#    - BELONGS_TO: 产品 -> 品类
#    - COMPETES_WITH: 同品类不同产品
#    - COMPLEMENTARY_TO: 跑步手表 -> 运动耳机（跨品类互补）
#    - REVIEWED: 从 interactions 数据构建（带rating）
#    - PROMOTES: 活动 -> 产品
#    - PROMOTED_THROUGH: 活动 -> 渠道
# 4. 打印统计信息

raise NotImplementedError

## TODO 4：知识图谱查询与分析

用 networkx 图算法执行知识图谱查询：
1. 最短路径：找客户C001到产品P006的 shortest_path
2. 邻居查询：找产品P001的所有直接邻居（1跳）
3. 中心性分析：计算 degree_centrality 和 betweenness_centrality，找出核心节点
4. 社区发现：用 louvain_communities 检测社区结构

**提示**：用 `nx.shortest_path()`, `G.neighbors()`, `nx.degree_centrality()`, `nx.betweenness_centrality()`, `nx.community.louvain_communities()`

In [ ]:
# TODO: 你的代码
# 1. 最短路径: C001 -> P006
# 2. P001的所有直接邻居
# 3. 度中心性 + 介数中心性 (Top 5)
# 4. Louvain社区发现

raise NotImplementedError

## TODO 5：GraphRAG混合检索

实现GraphRAG混合检索（向量检索+图谱多跳检索）：
1. 向量检索：用sentence-transformers做语义检索
2. 图谱检索：从检索到的产品出发，沿KG关系链做多跳检索（找到互补品/竞品/同品类）
3. 结果融合：合并向量检索和图谱检索结果，去重排序
4. 查询 '跑步爱好者需要什么装备'，对比纯向量检索 vs 混合检索结果

**提示**：图谱检索用 `G.neighbors()` 和 `G.edges(data=True)` 沿 COMPLEMENTARY_TO/COMPETES_WITH 关系做多跳

In [ ]:
# TODO: 你的代码
# 1. 向量检索函数（复用TODO2）
# 2. 图谱多跳检索函数 graph_search(product_nodes, G, hops=2)
# 3. 混合检索函数 graph_rag_search(query, model, product_embeddings, G, top_k=3)
# 4. 查询 '跑步爱好者需要什么装备'，对比纯向量 vs 混合检索

raise NotImplementedError

## TODO 6：GraphRAG vs 传统向量RAG效果对比

对比GraphRAG混合检索与传统向量RAG在多跳营销问答上的效果：
1. 设计5个营销查询（含简单事实型+多跳关系型）
2. 对每个查询，分别用纯向量检索和GraphRAG混合检索
3. 评估召回率：手动标注期望结果，计算两种方法的recall@5
4. 分析：GraphRAG在哪些类型问题上显著优于传统向量RAG？

**提示**：多跳关系型问题（如'买了跑步手表的客户还买了什么'）应该体现GraphRAG的优势

In [ ]:
# TODO: 你的代码
# 1. 设计5个查询 + 期望结果
# 2. 纯向量检索 vs GraphRAG混合检索
# 3. 计算 recall@5
# 4. 打印对比结果

raise NotImplementedError

## 完成检查

- [ ] TODO1: 数据预处理与特征工程
- [ ] TODO2: 向量化表示与语义检索
- [ ] TODO3: 营销知识图谱构建
- [ ] TODO4: 知识图谱查询与分析
- [ ] TODO5: GraphRAG混合检索
- [ ] TODO6: GraphRAG vs 传统RAG对比

完成后查看 `solution.ipynb` 对比答案。